## 03 - Acquisition Functions and the Query Pipeline

This notebook digs into how the model decides *where* to query next. The surrogate model gives us predictions and uncertainty estimates — the acquisition function is what turns those into a single recommended point.

I also want to look at how kernel selection actually plays out across the 8 functions now that we have W6 data, and compare what UCB vs EI would have suggested for each function.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.surrogate import (
    load_all_data, fit_gp, optimize_acquisition, propose_week_queries,
    _ucb, _ei, _auto_select_acquisition, FUNCTION_DIMENSIONS
)

plt.style.use('seaborn-v0_8-whitegrid')
PROJECT_ROOT = Path('..').resolve()

### 1. Which Kernel Did Each Function Pick?

Each time we fit a model, we try four different kernel shapes and pick the one that scores best on the data. Let's see which one won for each function with the current dataset (baseline + W1–W6).

In [ ]:
# Fit models and record which kernel was selected
kernel_names = {
    'Matern(nu=2.5)': 'Matern 5/2',
    'Matern(nu=1.5)': 'Matern 3/2',
    'RBF':            'RBF',
    'RationalQuadratic': 'RationalQuadratic',
}

gp_models = {}
kernel_results = []

for fid in range(1, 9):
    gp, X, y = fit_gp(fid)
    gp_models[fid] = (gp, X, y)

    # The fitted kernel is: ConstantKernel * <base_kernel> + WhiteKernel
    # kernel_.k1.k2 gives us the base kernel
    base_kernel = gp.kernel_.k1.k2
    kname = type(base_kernel).__name__
    lml   = round(gp.log_marginal_likelihood_value_, 2)

    kernel_results.append({
        'Function': fid,
        'Dims': FUNCTION_DIMENSIONS[fid],
        'Kernel': kernel_names.get(str(base_kernel), kname),
        'Log-Marginal Likelihood': lml,
        'Data Points': len(y),
    })

kr_df = pd.DataFrame(kernel_results)
print('Kernel Selection Results (W1-W6 data)')
print('=' * 60)
print(kr_df.to_string(index=False))

Most functions still prefer a smoother kernel (RBF or Matern 5/2). The interesting cases are the ones that switch — if a function picks Matern 3/2 it suggests the landscape has sharper features that a smooth kernel would miss.

It's worth checking whether these selections are stable or whether they keep changing as more data comes in.

### 2. UCB vs EI — What's the Difference?

Both acquisition functions use the same GP predictions (mean and uncertainty), but they use them differently:

- **UCB** = mean + β × uncertainty. It actively rewards going to uncertain regions, even if the predicted value isn't great.
- **EI** = expected improvement over the current best. It only rewards going somewhere if the predicted distribution suggests a real chance of beating what we already have.

For the 2D functions (F1 and F2) we can plot both scoring surfaces side by side and see exactly where each one would send the next query.

In [ ]:
# UCB vs EI scoring surfaces for F1 and F2
grid_res = 80
x1g = np.linspace(0.001, 0.999, grid_res)
x2g = np.linspace(0.001, 0.999, grid_res)
X1, X2 = np.meshgrid(x1g, x2g)
X_grid  = np.column_stack([X1.ravel(), X2.ravel()])

for fid in [1, 2]:
    gp, X_obs, y_obs = gp_models[fid]
    y_best = y_obs.max()

    ucb_vals = _ucb(gp, X_grid).reshape(grid_res, grid_res)
    ei_vals  = _ei(gp, X_grid, y_best).reshape(grid_res, grid_res)

    x_ucb = optimize_acquisition(fid, gp=gp, X=X_obs, y=y_obs, acq='ucb')
    x_ei  = optimize_acquisition(fid, gp=gp, X=X_obs, y=y_obs, acq='ei')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    cf = ax.contourf(X1, X2, ucb_vals, levels=30, cmap='plasma')
    plt.colorbar(cf, ax=ax, label='UCB score')
    ax.scatter(X_obs[:, 0], X_obs[:, 1], c='white', edgecolors='black', s=40, zorder=5)
    ax.scatter(x_ucb[0], x_ucb[1], marker='*', c='red', s=250, zorder=6, label='UCB proposal')
    ax.set_title(f'F{fid} — UCB scoring surface')
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.legend()

    ax = axes[1]
    cf = ax.contourf(X1, X2, ei_vals, levels=30, cmap='viridis')
    plt.colorbar(cf, ax=ax, label='EI score')
    ax.scatter(X_obs[:, 0], X_obs[:, 1], c='white', edgecolors='black', s=40, zorder=5)
    ax.scatter(x_ei[0], x_ei[1], marker='*', c='orange', s=250, zorder=6, label='EI proposal')
    ax.set_title(f'F{fid} — EI scoring surface')
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.legend()

    plt.suptitle(f'Function {fid}: UCB vs EI', fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f'F{fid} — UCB proposes: [{x_ucb[0]:.4f}, {x_ucb[1]:.4f}]')
    print(f'F{fid} — EI  proposes: [{x_ei[0]:.4f},  {x_ei[1]:.4f}]')
    print()

### 3. Auto-Selection — Let the Model Decide

From Week 7 onward, when no explicit strategy is given, the pipeline runs both UCB and EI and picks whichever proposes the point with the higher predicted mean. The idea is to avoid baking in a blanket assumption ("always UCB" was the old default) and instead let the data guide the choice.

An explicit `acq_map` dict still overrides this — used for functions where we have a strong prior about what strategy to use.

In [ ]:
# Show auto-selection results for all 8 functions
print('Auto-Selection: UCB vs EI per function (current data)')
print('=' * 65)

for fid in range(1, 9):
    gp, X_obs, y_obs = gp_models[fid]
    x_star, chosen = _auto_select_acquisition(fid, gp, X_obs, y_obs)
    mu_star = float(gp.predict(x_star.reshape(1, -1))[0])
    print(f'F{fid}: auto-selected {chosen.upper():3s}  |  predicted mean at proposal: {mu_star:.4f}')

### 4. End-to-End Pipeline

Here's the full flow from raw data to a submitted query:

1. Load all data for a function (baseline + all weekly results)
2. Fit four different GP kernels, pick the one with the best score
3. Run the acquisition function (UCB, EI, or auto-select) over the full search space
4. Return the point that scores highest

Below I run this end-to-end for F4 — the function that had a breakthrough in W5 and then fell back in W6. The EI candidate for W7 should be trying to recover toward that W5 region.

In [ ]:
# End-to-end for F4
fid = 4
gp, X_obs, y_obs = fit_gp(fid)

print(f'F4: {len(y_obs)} data points')
print(f'F4: best observed value = {y_obs.max():.4f}')
print()

# Show the top 5 observed points
top_idx = np.argsort(y_obs)[-5:][::-1]
print('Top 5 observed points:')
for i in top_idx:
    coords = ', '.join(f'{v:.3f}' for v in X_obs[i])
    print(f'  [{coords}]  →  {y_obs[i]:.4f}')
print()

# W7 EI proposal
x_w7 = optimize_acquisition(fid, gp=gp, X=X_obs, y=y_obs, acq='ei')
mu_w7, sigma_w7 = gp.predict(x_w7.reshape(1, -1), return_std=True)
coords_w7 = ', '.join(f'{v:.6f}' for v in x_w7)
print(f'W7 EI proposal: [{coords_w7}]')
print(f'Predicted mean: {mu_w7[0]:.4f}  |  Uncertainty: {sigma_w7[0]:.4f}')

### 5. Heuristic vs Model-Driven — A Quick Comparison

It's easy to forget how different the early queries were. Weeks 1–4 used a simple cluster-centre heuristic — no model, just geometric spread and intuition. The surrogate model from Week 5 onward is fundamentally different: it predicts the full surface and scores every candidate point.

The table below shows the best result per function across the two eras.

In [ ]:
# Best heuristic result (W1-W4) vs best model-driven result (W5-W6) per function
heuristic_weeks = [1, 2, 3, 4]
model_weeks     = [5, 6]

rows = []
for fid in range(1, 9):
    dim    = FUNCTION_DIMENSIONS[fid]
    x_cols = [f'x{i}' for i in range(1, dim + 1)]

    h_best, m_best = -np.inf, -np.inf

    for wk_list, store in [(heuristic_weeks, 'h'), (model_weeks, 'm')]:
        for w in wk_list:
            path = PROJECT_ROOT / 'data' / 'weekly' / f'week_{w:02d}.csv'
            wk   = pd.read_csv(path)
            row  = wk[wk['function_id'] == fid]
            if row.empty:
                continue
            row   = row.iloc[0]
            y_val = row['y'] if pd.notna(row['y']) else float(row.dropna().iloc[-1])
            if store == 'h':
                h_best = max(h_best, y_val)
            else:
                m_best = max(m_best, y_val)

    rows.append({
        'Function': fid,
        'Best (W1-W4 heuristic)': round(h_best, 4) if h_best > -np.inf else 'n/a',
        'Best (W5-W6 model)':     round(m_best, 4) if m_best > -np.inf else 'n/a',
        'Model better?': 'YES' if m_best > h_best else '',
    })

cmp_df = pd.DataFrame(rows)
print('Heuristic era (W1-W4) vs Model era (W5-W6)')
print('=' * 65)
print(cmp_df.to_string(index=False))

---

### 6. W7 Update — Are the Kernel Selections Stable?

One thing I wanted to track is whether the kernel each function picks changes as more data comes in. With W7 data now included, let's re-run kernel selection and compare against the W6 selections recorded in the model card.

In [ ]:
# W6 kernel selections (from model card) vs W7 re-run
w6_kernels = {1:'RBF', 2:'RBF', 3:'RBF', 4:'RationalQuadratic', 5:'RBF', 6:'Matern', 7:'RBF', 8:'Matern'}

kernel_names = {
    'Matern(nu=2.5)':    'Matern 5/2',
    'Matern(nu=1.5)':    'Matern 3/2',
    'RBF':               'RBF',
    'RationalQuadratic': 'RationalQuadratic',
}

print('Kernel stability check: W6 vs W7')
print('=' * 55)
print(f'{"F":<4} {"W6 kernel":<22} {"W7 kernel":<22} {"Changed?"}')
print('-' * 55)

for fid in range(1, 9):
    gp, X, y = fit_gp(fid)
    base_kernel = gp.kernel_.k1.k2
    kname = kernel_names.get(type(base_kernel).__name__, type(base_kernel).__name__)
    changed = 'YES' if kname != w6_kernels[fid] else ''
    print(f'F{fid:<3} {w6_kernels[fid]:<22} {kname:<22} {changed}')

If kernels are changing week to week with a single new data point, it suggests the selection isn't stable yet — the model doesn't have enough data to be confident in a shape. If they stay the same, it's a good sign the fits are settling.

### 7. W8 Query Generation — How the Decisions Were Made

This section shows the full pipeline for W8: run the surrogate, check for duplicates, apply overrides where needed. This is the actual process, not a cleaned-up demo.

In [ ]:
# Step 1: Run surrogate with W8 acquisition map
# Reasoning:
#   F1, F2, F3 — UCB: haven't found a good region yet
#   F4 — EI: trying to exploit near the W5 winner
#   F5 — EI: boundary probing (may need manual override)
#   F6 — EI: two consecutive new bests, keep exploiting
#   F7 — UCB: EI failed in W7, switch back to searching
#   F8 — EI: new best in W7, keep going

acq_map = {1:'ucb', 2:'ucb', 3:'ucb', 4:'ei', 5:'ei', 6:'ei', 7:'ucb', 8:'ei'}
raw_proposals = propose_week_queries(acq_map=acq_map)

print('Raw surrogate proposals for W8:')
print('=' * 65)
for fid, (vec, acq) in raw_proposals.items():
    formatted = ', '.join(f'{v:.6f}' for v in vec)
    print(f'F{fid} ({acq.upper()}): [{formatted}]')

In [ ]:
# Step 2: Duplicate check — compare every proposal against baseline + W1-W7
NEAR_DUP_THRESH = 0.01

flags = {}
for fid, (vec, acq) in raw_proposals.items():
    dim    = FUNCTION_DIMENSIONS[fid]
    x_cols = [f'x{i}' for i in range(1, dim + 1)]
    prop   = np.array(vec)
    issues = []

    bl = pd.read_csv(PROJECT_ROOT / 'data' / 'baseline' / f'function_{fid}_baseline.csv')
    all_X = [('baseline', bl[x_cols].values)]

    for w in range(1, 8):
        wk  = pd.read_csv(PROJECT_ROOT / 'data' / 'weekly' / f'week_{w:02d}.csv')
        row = wk[wk['function_id'] == fid]
        if row.empty: continue
        x_vals = row.iloc[0][x_cols].values.astype(float)
        if not np.any(np.isnan(x_vals)):
            all_X.append((f'W{w}', x_vals.reshape(1, -1)))

    for source, X in all_X:
        for x in X:
            diff = np.abs(x - prop)
            if np.all(diff < 1e-4):
                issues.append(f'EXACT DUPLICATE of {source}')
            elif np.all(diff < NEAR_DUP_THRESH):
                issues.append(f'near-duplicate of {source} (max_diff={diff.max():.4f})')

    flags[fid] = issues
    status = ', '.join(issues) if issues else 'OK'
    print(f'F{fid}: {status}')

F4 came back as a near-duplicate of W7. The EI candidate was within 0.003 of last week's query — essentially the same point. UCB returned the same thing. The surrogate is stuck: it keeps finding the same local region promising because that's where most of the recent data points are clustered.

F5 also needs a manual override. The EI proposal [0.999, 0.001, 0.001, 0.999] has x2 and x3 both at the lower boundary, which contradicts what we just learned in W7 (x3=0.001 dropped output from 8585 to 4399). The model is sending EI toward high-uncertainty regions rather than the known good corner.

In [ ]:
# Step 3: Apply manual overrides for F4 and F5, finalise W8 queries

# F4: surrogate stuck near W7 point. Manual point nudges x3 back toward
#     the W5 winner (x3=0.354) which produced the best F4 result so far.
#     Checked clean against all prior data.
overrides = {
    4: [0.425000, 0.418000, 0.360000, 0.435000],  # near W5 winner, x3 toward 0.354
    5: [0.999000, 0.999000, 0.999000, 0.001000],  # probe x4 (already tested x3 in W7)
}

final_proposals = {}
for fid, (vec, acq) in raw_proposals.items():
    if fid in overrides:
        final_proposals[fid] = (overrides[fid], 'manual')
    else:
        final_proposals[fid] = (vec, acq)

print('Final W8 queries (after overrides):')
print('=' * 65)
for fid, (vec, acq) in final_proposals.items():
    formatted = ', '.join(f'{v:.6f}' for v in vec)
    note = ' <- OVERRIDE' if fid in overrides else ''
    print(f'F{fid} ({acq.upper()}): [{formatted}]{note}')

The overrides show a limitation of the surrogate that becomes more visible as the project progresses. When a function has a narrow good region and we keep querying near it, the model gets increasingly confident about that area and EI stops exploring — it finds the same point optimal every time. The duplicate check is what catches this before we waste a query.

For F5, the situation is different. The model isn't stuck — it's sending EI toward genuinely uncertain regions, but those regions are likely poor based on what we know. This is a case where domain knowledge (x3 needs to be high) overrides what the model suggests. The systematic probing approach — testing one dimension at a time away from the known best corner — is more useful here than letting the model roam freely.

## Week 8 Results and W9 Query Generation

W8 closed the loop on F5 and gave more evidence that F4's good region is narrow but real.

In [ ]:
# W8 outcomes vs expectations
import pandas as pd

w8_summary = [
    {"Function": "F4", "Query": "[0.425, 0.418, 0.360, 0.435]", "Result": 0.549, "Best": 0.696, "Note": "Recovering — manual x3=0.354 working"},
    {"Function": "F5", "Query": "[0.999, 0.999, 0.999, 0.001]", "Result": 4399.1, "Best": 8585.3, "Note": "x4 confirmed critical"},
    {"Function": "F6", "Query": "[0.464, 0.290, 0.580, 0.621, 0.001]", "Result": -0.458, "Best": -0.328, "Note": "EI overshot, back to UCB"},
    {"Function": "F7", "Query": "[0.001, 0.336, 0.999, 0.253, 0.335, 0.629]", "Result": 1.140, "Best": 1.365, "Note": "UCB holding steady"},
    {"Function": "F8", "Query": "[0.182, 0.116, ..., 0.001]", "Result": 9.874, "Best": 9.930, "Note": "x8=0.001 hurt; EI drifting"},
]
print(pd.DataFrame(w8_summary).to_string(index=False))


**F5 corner map is now complete:**

| Probe | x1 | x2 | x3 | x4 | Result |
|-------|----|----|----|----|--------|
| W6 (baseline) | 0.999 | 0.999 | 0.999 | 0.999 | 8585.3 |
| W7 (x3 probe) | 0.999 | 0.999 | 0.001 | 0.999 | 4399 |
| W8 (x4 probe) | 0.999 | 0.999 | 0.999 | 0.001 | 4399 |
| W9 (x1 probe) | 0.001 | 0.999 | 0.999 | 0.999 | ? |

Three of the four corners have been systematically tested. W9 completes the picture for x1.

In [ ]:
# W9 query generation — same pipeline as W8
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from src.surrogate import propose_week_queries

acq_map = {1:"ucb", 2:"ucb", 3:"ucb", 4:"ei", 5:"ucb", 6:"ucb", 7:"ucb", 8:"ei"}
raw = propose_week_queries(acq_map=acq_map)

# Overrides: F5 exact dup of W6, F4 near-dup of W7
final = {}
for fid, (x, acq) in raw.items():
    final[fid] = x
final[4] = [0.440, 0.430, 0.354, 0.410]  # manual: keep x3 at sweet spot
final[5] = [0.001, 0.999, 0.999, 0.999]  # probe x1

for fid, x in final.items():
    print(f"F{fid}: {[round(v,6) for v in x]}")


## Week 9 Results and W10 Query Generation

W9 gave two new bests and completed the F5 corner map. W10 shifts to exploitation for F7 and F8, and probes the last F5 corner dimension.

In [ ]:
# W9 outcomes summary
import pandas as pd

w9_summary = [
    {"Function": "F5", "Result": 4399.1, "Best": 8585.3, "Note": "x1=0.001 drops to 4399 — corner map complete"},
    {"Function": "F7", "Result": 1.5363, "Best": 1.5363, "Note": "NEW BEST — beats baseline 1.365 for first time"},
    {"Function": "F8", "Result": 9.9720, "Best": 9.9720, "Note": "NEW BEST"},
    {"Function": "F2", "Result": 0.4802, "Best": 0.611,  "Note": "Best weekly result, baseline still leads"},
    {"Function": "F4", "Result": 0.3932, "Best": 0.696,  "Note": "Drifting — manual near x3=0.354 not holding"},
    {"Function": "F6", "Result": -0.6349,"Best": -0.328, "Note": "Worst F6 result yet — UCB went badly"},
]
print(pd.DataFrame(w9_summary).to_string(index=False))


**F5 corner map summary:**

| Probe | x1 | x2 | x3 | x4 | Result |
|-------|----|----|----|----|--------|
| W6 (all high) | 0.999 | 0.999 | 0.999 | 0.999 | 8585.3 |
| W7 (x3 low) | 0.999 | 0.999 | 0.001 | 0.999 | 4399 |
| W8 (x4 low) | 0.999 | 0.999 | 0.999 | 0.001 | 4399 |
| W9 (x1 low) | 0.001 | 0.999 | 0.999 | 0.999 | 4399 |
| W10 (x2 low) | 0.999 | 0.001 | 0.999 | 0.999 | ? |

Each dimension matters — pulling any one to 0.001 halves the output. W10 tests x2 to complete the picture.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from src.surrogate import propose_week_queries

# W10 strategy: F7 and F8 on EI (new bests to exploit), F6 back to EI,
# F4 EI with updated data, F1/F2/F3 UCB, F5 manual x2 probe
acq_map = {1:"ucb", 2:"ucb", 3:"ucb", 4:"ei", 5:"ucb", 6:"ei", 7:"ei", 8:"ei"}
raw = propose_week_queries(acq_map=acq_map)

# F5 will duplicate W6 corner — override to probe x2
final = {fid: x for fid, (x, _) in raw.items()}
final[5] = [0.999, 0.001, 0.999, 0.999]

print("W10 final queries:")
for fid, x in final.items():
    print(f"  F{fid}: {[round(v, 6) for v in x]}")


## Week 10 Results and W11 Query Generation

W10 confirmed the F5 corner map and delivered F2's first baseline-beating result. W11 shifts to exploitation for F2 and breaks F4 out of its near-duplicate loop.

In [ ]:
# W10 outcomes summary
import pandas as pd

w10_summary = [
    {'Function': 'F2', 'Result': 0.6937, 'Best': 0.6937, 'Note': 'NEW BEST — beats baseline 0.611 for first time'},
    {'Function': 'F5', 'Result': 4399.1, 'Best': 8585.3, 'Note': 'x2=0.001 confirms corner map complete'},
    {'Function': 'F8', 'Result': 9.9522, 'Best': 9.9720, 'Note': 'Slight slip from W9 best'},
    {'Function': 'F7', 'Result': 1.0221, 'Best': 1.5363, 'Note': 'EI dropped — switching back to UCB'},
    {'Function': 'F6', 'Result': -0.3804,'Best': -0.328, 'Note': 'Recovery from W9 disaster, still below W7 best'},
    {'Function': 'F4', 'Result': 0.2952, 'Best': 0.696,  'Note': 'Surrogate stuck in near-dup loop'},
    {'Function': 'F3', 'Result': -0.4348,'Best': -0.035, 'Note': 'Worst ever — UCB found a bad corner'},
]
print(pd.DataFrame(w10_summary).to_string(index=False))

**F5 corner map — complete:**

| Probe | x1 | x2 | x3 | x4 | Result |
|-------|----|----|----|----|--------|
| W6 (all high) | 0.999 | 0.999 | 0.999 | 0.999 | 8585.3 |
| W7 (x3 low)   | 0.999 | 0.999 | 0.001 | 0.999 | 4399 |
| W8 (x4 low)   | 0.999 | 0.999 | 0.999 | 0.001 | 4399 |
| W9 (x1 low)   | 0.001 | 0.999 | 0.999 | 0.999 | 4399 |
| W10 (x2 low)  | 0.999 | 0.001 | 0.999 | 0.999 | 4399 |

Every dimension is critical. The optimum is definitively [0.999, 0.999, 0.999, 0.999]. No more probing needed for F5 — W11 tests how sharply the peak falls by trying [0.980, 0.999, 0.999, 0.999].

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.surrogate import propose_week_queries, fit_gp, optimize_acquisition
import numpy as np

# W11 strategy:
# F1, F2, F3 — UCB (auto-select chose UCB for F2; explores whether x2=0.999 is essential)
# F4 — manual escape: model stuck in near-dup loop; keep x3=0.354 but shift x1,x2,x4
# F5 — manual: [0.980,0.999,0.999,0.999] probes peak sharpness (avoids W6 exact dup)
# F6, F8 — EI
# F7 — UCB override (EI dropped from 1.536 to 1.022 in W10)

# Get UCB for F7 explicitly
gp7, X7, y7 = fit_gp(7)
x_ucb7 = optimize_acquisition(7, gp=gp7, X=X7, y=y7, acq='ucb')

acq_map = {1:'ucb', 2:'ucb', 3:'ucb', 6:'ei', 8:'ei'}
raw = propose_week_queries(acq_map=acq_map)

final = {fid: x for fid, (x, _) in raw.items()}
final[4] = [0.430, 0.390, 0.354, 0.460]   # manual escape
final[5] = [0.980, 0.999, 0.999, 0.999]   # peak sharpness probe
final[7] = x_ucb7.tolist()                 # UCB override

print('W11 final queries:')
for fid, x in final.items():
    print(f'  F{fid}: {[round(v, 6) for v in x]}')

## Week 11 Results and W12 Query Generation

W11 gave F7 a new high and revealed useful gradient information for F5. W12 focuses on exploiting F7's new best and correcting the F2 model's bad proposal.

In [ ]:
import pandas as pd

w11_summary = [
    {'Function': 'F7', 'Result': 1.6845, 'Best': 1.6845, 'Note': 'NEW BEST — UCB beats W9 1.536'},
    {'Function': 'F5', 'Result': 8233.7, 'Best': 8585.3, 'Note': 'x1=0.980 gives 8233 — peak is gradual (~4% drop)'},
    {'Function': 'F3', 'Result': -0.1013, 'Best': -0.035,  'Note': 'Recovery from W10 worst, still below baseline'},
    {'Function': 'F8', 'Result': 9.9033, 'Best': 9.9720,  'Note': 'Slight slip — x8=0.999 unhelpful'},
    {'Function': 'F2', 'Result': 0.4682, 'Best': 0.6937,  'Note': 'x2=0.654 confirmed x2=0.999 essential'},
    {'Function': 'F6', 'Result': -0.4250,'Best': -0.328,  'Note': 'EI not recovering'},
    {'Function': 'F4', 'Result': -0.2593,'Best': 0.696,   'Note': 'Manual escape went negative — too far from good region'},
]
print(pd.DataFrame(w11_summary).to_string(index=False))

**Note on F2 model failure:** Both UCB and EI proposed x2=0.001 for W12, despite W10 and W11 data clearly showing x2=0.999 drives the best results. This is a case where the GP surface is being pulled by the wider dataset (most F2 baseline points have lower x2) and overriding the recent signal. Manual override forces the query to [0.780, 0.999].

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.surrogate import propose_week_queries, fit_gp, optimize_acquisition
import numpy as np

# W12 strategy:
# F7 — EI: exploit new best 1.685
# F2 — manual [0.780, 0.999]: model proposes x2=0.001 despite evidence; override
# F5 — manual [0.999, 0.980, 0.999, 0.999]: probe x2 gradient
# F4, F6, F8 — EI; F1, F3 — UCB

acq_map = {1:'ucb', 3:'ucb', 4:'ei', 6:'ei', 7:'ei', 8:'ei'}
raw = propose_week_queries(acq_map=acq_map)

final = {fid: x for fid, (x, _) in raw.items()}
final[2] = [0.780, 0.999]          # manual: x2=0.999 confirmed
final[5] = [0.999, 0.980, 0.999, 0.999]  # manual: x2 gradient probe

print('W12 final queries:')
for fid, x in final.items():
    print(f'  F{fid}: {[round(v, 6) for v in x]}')

## Week 12 Results and W13 Query Generation (Final Round)

W12 gave F7 another new best and confirmed the F5 peak gradient is symmetric. W13 is the last query submission — every function targets its best known region.

In [ ]:
import pandas as pd

w12_summary = [
    {'Function': 'F7', 'Result': 1.7562, 'Best': 1.7562, 'Note': 'NEW BEST — third consecutive improvement'},
    {'Function': 'F5', 'Result': 8233.7, 'Best': 8585.3, 'Note': 'x2=0.980 gives 8233 — gradient symmetric with x1 probe'},
    {'Function': 'F3', 'Result': -0.0375,'Best': -0.035,  'Note': 'Closest ever to baseline -0.035'},
    {'Function': 'F4', 'Result': 0.3810, 'Best': 0.696,   'Note': 'Positive recovery, still below W5 best'},
    {'Function': 'F6', 'Result': -0.3529,'Best': -0.328,  'Note': 'Best since W7, recovery continuing'},
    {'Function': 'F8', 'Result': 9.9529, 'Best': 9.972,   'Note': 'Small improvement from W11'},
    {'Function': 'F2', 'Result': 0.1281, 'Best': 0.6937,  'Note': 'x1=0.780 too far — x1 sweet spot is narrow near 0.716'},
]
print(pd.DataFrame(w12_summary).to_string(index=False))

**F5 peak gradient summary:**

| Query | x1 | x2 | x3 | x4 | Result |
|-------|----|----|----|----|--------|
| W6 (full corner) | 0.999 | 0.999 | 0.999 | 0.999 | 8585.3 |
| W11 (x1=0.980) | 0.980 | 0.999 | 0.999 | 0.999 | 8233.7 |
| W12 (x2=0.980) | 0.999 | 0.980 | 0.999 | 0.999 | 8233.7 |

The gradient is symmetric — a shift of 0.019 in either x1 or x2 away from 0.999 causes the same ~4% output drop. The peak is broad near the corner.

W13 submits [0.999, 0.999, 0.985, 0.999] — the closest achievable point to the confirmed optimum that has not already been submitted.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.surrogate import propose_week_queries, load_all_data
import numpy as np

# W13 — FINAL ROUND. All functions target best known regions.
# F7: EI exploit 1.756. F2: UCB auto near x1=0.700, x2=0.999.
# F4: manual [0.460, 0.380, 0.354, 0.400] — keep x3=0.354, avoid dup loop.
# F5: manual [0.999, 0.999, 0.985, 0.999] — closest clean point to confirmed optimum.
# F3, F6, F8: model-driven EI/auto.

acq_map = {1:'ucb', 2:'ucb', 3:'ei', 4:'ei', 6:'ei', 7:'ei', 8:'ei'}
raw = propose_week_queries(acq_map=acq_map)

final = {fid: x for fid, (x, _) in raw.items()}
final[4] = [0.460, 0.380, 0.354, 0.400]        # manual: escape dup loop
final[5] = [0.999, 0.999, 0.985, 0.999]        # manual: nearest clean point to optimum

print('W13 FINAL queries:')
for fid, x in final.items():
    print(f'  F{fid}: {[round(v, 6) for v in x]}')

## Week 13 Results (Final — No More Queries)

W13 was the last submission. This section records what came back and closes out the pipeline. No W14 queries to generate.

In [ ]:
import pandas as pd

w13_summary = [
    {'Function': 'F5', 'W13 Result': 8323.6, 'All-time Best': 8585.3, 'Best Week': 'W6', 'Note': 'x3=0.985 probe — best since W6'},
    {'Function': 'F2', 'W13 Result': 0.665,  'All-time Best': 0.694,  'Best Week': 'W10', 'Note': 'Close to all-time best'},
    {'Function': 'F8', 'W13 Result': 9.965,  'All-time Best': 9.972,  'Best Week': 'W9',  'Note': 'Near all-time best'},
    {'Function': 'F7', 'W13 Result': 1.391,  'All-time Best': 1.756,  'Best Week': 'W12', 'Note': 'EI overshot on final try'},
    {'Function': 'F6', 'W13 Result': -0.422, 'All-time Best': -0.328, 'Best Week': 'W7',  'Note': 'Slight slip from W12'},
    {'Function': 'F3', 'W13 Result': -0.069, 'All-time Best': -0.035, 'Best Week': 'Baseline', 'Note': 'Never beat baseline'},
    {'Function': 'F4', 'W13 Result': -0.448, 'All-time Best': 0.696,  'Best Week': 'W5',  'Note': 'Negative again — region too narrow'},
    {'Function': 'F1', 'W13 Result': -9.27e-16, 'All-time Best': 1.6e-15, 'Best Week': 'W4', 'Note': 'Near-zero as always'},
]

df_w13 = pd.DataFrame(w13_summary)
print(df_w13.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Final best results vs baseline comparison
functions = ['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8']
best_weekly = [1.6e-15, 0.694, -0.038, 0.696, 8585.3, -0.328, 1.756, 9.972]
baselines =   [7.7e-16, 0.611, -0.035, -4.026, 1088.9, -0.714, 1.365, 9.598]

# Normalise for comparison: improvement ratio (weekly/baseline)
# Exclude F1 and F5 (extreme scales), show F2-F4, F6-F8
plot_fns   = ['F2', 'F3', 'F4', 'F6', 'F7', 'F8']
plot_best  = [0.694, -0.038, 0.696, -0.328, 1.756, 9.972]
plot_base  = [0.611, -0.035, -4.026, -0.714, 1.365, 9.598]

x = np.arange(len(plot_fns))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width/2, plot_best, width, label='Best weekly', color='steelblue')
ax.bar(x + width/2, plot_base, width, label='Baseline best', color='lightgrey', edgecolor='grey')
ax.set_xticks(x)
ax.set_xticklabels(plot_fns)
ax.set_title('Final best weekly result vs baseline (F2-F8, excl F1 and F5)')
ax.set_ylabel('Output value')
ax.legend()
plt.tight_layout()
plt.show()

### Final summary

**7 out of 8 functions beat the baseline.** F3 is the only exception — the baseline best of -0.035 was never surpassed, though W12 got within 0.003 of it.

The biggest wins came from F4 (baseline was -4.026, model found 0.696 in W5), F5 (baseline 1088.9, model found 8585.3 in W6), and F6 (baseline -0.714, model reached -0.328 in W7). These weren't incremental improvements — the surrogate found completely different regions of the input space.

F7 and F8 improved steadily over multiple weeks through a mix of UCB exploration and EI exploitation. F2 beat its baseline for the first time in W10 and held near that level through W13.

F4 proved the most frustrating — the W5 breakthrough of 0.696 was never reproduced, and the surrogate kept getting stuck in near-duplicate proposals around the same narrow neighbourhood. The good region exists but it's too narrow to exploit reliably with one query per week.